# Triggers and Events

In [7]:
import os
from dotenv import load_dotenv

import pandas as pd
import sqlalchemy

In [8]:
load_dotenv()

db_host = os.environ.get("db_host")
db_user = os.environ.get("db_user")
db_password = os.environ.get("db_password")

In [9]:
engine = sqlalchemy.create_engine(f"mysql+pymysql://{db_user}:{db_password}@{db_host}:3306/sql_invoicing")

In [10]:
pd.read_sql("SHOW TABLES", con= engine)

,Tables_in_sql_invoicing
0,clients
1,invoices
2,payment_methods
3,payments


## Triggers
Trigger is a block of SQL code that automatically gets executed before or after an insert, update or delete statement.

In [20]:
query = sqlalchemy.text("""
CREATE TRIGGER payment_after_insert
	after insert on payments
    for each row
begin
	update invoices
    set payment_total = payment_total + new.amount
    where invoice_id = new.invoice_id;
end
""")

with engine.begin() as conn:
    conn.execute(query)                        

Triggers can be executed BEFORE or AFTER a database event such as `INSERT`, `UPDATE`, or `DELETE`. A `BEFORE` trigger runs before the operation is performed, while an `AFTER` trigger runs after the operation has successfully completed.


In SQL triggers, `NEW` and `OLD` are special pseudo-records used to access row data during `INSERT`, `UPDATE`, and `DELETE` operations.
- `NEW` refers to the row's values after the change (or the row being inserted).
- `OLD` refers to the row's values before the change. It is commonly used in UPDATE and DELETE triggers to compare old and new values or keep audit logs

In [21]:
query = """
SHOW TRIGGERS
"""

pd.read_sql(query, con= engine)

,Trigger,Event,Table,Statement,Timing,Created,sql_mode,Definer,character_set_client,collation_connection,Database Collation
0,payment_after_insert,INSERT,payments,begin\n\tupdate invoices\n set payment_tota...,AFTER,2026-06-24 20:40:55.490,"ONLY_FULL_GROUP_BY,STRICT_TRANS_TABLES,NO_ZERO...",root@localhost,utf8mb4,utf8mb4_0900_ai_ci,utf8mb4_0900_ai_ci


In [22]:
query = sqlalchemy.text("""
DROP TRIGGER IF EXISTS payment_after_insert;
""")

with engine.begin() as conn:
    conn.execute(query) 

## Events
Event is a task (or block of SQL code) that gets executed according to a schedule

In [27]:
query = sqlalchemy.text("""
CREATE EVENT yearly_delete_stale_audit_rows
ON SCHEDULE EVERY 1 YEAR STARTS '2019-01-01' ENDS '2030-01-01'
DO
BEGIN
    DELETE FROM payments_audit 
    WHERE action_date < NOW() - INTERVAL 1 YEAR;
END;
""")


with engine.begin() as conn:
    conn.execute(query) 

In [28]:
query = """
SHOW EVENTS
"""

pd.read_sql(query, con= engine)

,Db,Name,Definer,Time zone,Type,Execute at,Interval value,Interval field,Starts,Ends,Status,Originator,character_set_client,collation_connection,Database Collation
0,sql_invoicing,yearly_delete_stale_audit_rows,root@localhost,SYSTEM,RECURRING,None,1,YEAR,2019-01-01,2030-01-01,ENABLED,1,utf8mb4,utf8mb4_0900_ai_ci,utf8mb4_0900_ai_ci


In [29]:
query = sqlalchemy.text("""
DROP EVENT IF EXISTS yearly_delete_stale_audit_rows;
""")


with engine.begin() as conn:
    conn.execute(query) 